In [1]:
import os, random, math, warnings
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
 
import torchvision.models as tvm
import torchvision.transforms.functional as TF
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from tqdm.auto import tqdm
 
warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True


/Users/rahulmac/Documents/Projects/projects/Image_Forensics_v2/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [3]:
DATA_ROOT = Path("./dataset")          # root of GenImage dataset
CKPT_DIR  = Path("./checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

In [4]:
CFG = dict(
    img_size      = 224,
    batch_size    = 32,
    num_workers   = 4,
    epochs        = 30,
    lr            = 2e-4,
    weight_decay  = 1e-2,
    label_smooth  = 0.05,
    warmup_epochs = 3,
    grad_clip     = 1.0,
    amp           = True,
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)
print(f"Device: {CFG['device']}  |  AMP: {CFG['amp']}")

Device: cpu  |  AMP: True


In [7]:
# ── Cell 2 · Forensic Dataset & DataLoader (albumentations ≥ 1.4 compatible) ──
#
# Directory layout expected:
#   DATA_ROOT/
#     train/  val/  test/
#       real_pool/   → class 0
#       gan_pool/    → class 1
#       mj_pool/     → class 1
#       sd_pool/     → class 1
#
# Fallback: If train/val structure doesn't exist, pools at root are split 80/20

import cv2
AI_POOLS  = ["gan_pool", "mj_pool", "sd_pool"]
REAL_POOL = "real_pool"

_SZ = (CFG["img_size"], CFG["img_size"])   # albumentations ≥ 1.4 wants a tuple

# ── Social-media degradation augmentations (training only) ─────────────────────
TRAIN_TRANSFORMS = A.Compose([
    A.RandomResizedCrop(size=_SZ, scale=(0.7, 1.0)),          # ← size= tuple
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ImageCompression(quality_range=(40, 90), p=1.0),    # ← quality_range tuple
        A.Downscale(scale_range=(0.5, 0.9), p=1.0),           # ← scale_range tuple
    ], p=0.7),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=7, p=1.0),
        A.Sharpen(alpha=(0.1, 0.4), p=1.0),
    ], p=0.4),
    A.OneOf([
        A.GaussNoise(std_range=(0.02, 0.12), p=1.0),          # ← std_range replaces var_limit
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
    ], p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

VAL_TRANSFORMS = A.Compose([
    A.Resize(height=CFG["img_size"], width=CFG["img_size"]),  # height/width kwargs still valid
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class GenImageDataset(Dataset):
    EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

    def __init__(self, root: Path, split: str = "train", transform=None):
        self.transform = transform
        self.samples: list[tuple[Path, int]] = []

        split_dir = root / split                 # e.g.  ./dataset/train/
        
        # Try structured layout first (train/val subdirs exist)
        if (split_dir / REAL_POOL).exists():
            # ─ Structured layout: train/val/test subdirs ─
            real_dir = split_dir / REAL_POOL
            for p in sorted(real_dir.rglob("*")):
                if p.suffix.lower() in self.EXTENSIONS:
                    self.samples.append((p, 0))

            for pool in AI_POOLS:
                ai_dir = split_dir / pool
                if not ai_dir.exists():
                    continue
                for p in sorted(ai_dir.rglob("*")):
                    if p.suffix.lower() in self.EXTENSIONS:
                        self.samples.append((p, 1))
        else:
            # ─ Fallback: pools at root level, split 80/20 by split arg ─
            all_samples = []
            
            # Collect all images
            real_dir = root / REAL_POOL
            if real_dir.exists():
                for p in sorted(real_dir.rglob("*")):
                    if p.suffix.lower() in self.EXTENSIONS:
                        all_samples.append((p, 0))

            for pool in AI_POOLS:
                ai_dir = root / pool
                if not ai_dir.exists():
                    continue
                for p in sorted(ai_dir.rglob("*")):
                    if p.suffix.lower() in self.EXTENSIONS:
                        all_samples.append((p, 1))
            
            # Split into train/val using deterministic split
            random.seed(SEED)  # Ensure reproducibility
            random.shuffle(all_samples)
            split_idx = int(len(all_samples) * 0.8)
            
            if split == "train":
                self.samples = all_samples[:split_idx]
            elif split == "val":
                self.samples = all_samples[split_idx:]
            else:  # test
                self.samples = all_samples[split_idx:]  # Use val split for test

        random.shuffle(self.samples)
        labels = [s[1] for s in self.samples]
        n0, n1 = labels.count(0), labels.count(1)
        print(f"[{split}] real={n0:,}  AI={n1:,}  total={len(self.samples):,}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # cv2 faster than PIL for large datasets
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


def make_loaders(root: Path, cfg: dict):
    train_ds = GenImageDataset(root, "train", TRAIN_TRANSFORMS)
    val_ds   = GenImageDataset(root, "val",   VAL_TRANSFORMS)

    # Weighted sampler → handles class imbalance without oversampling real images
    labels   = [s[1] for s in train_ds.samples]
    counts   = np.bincount(labels).astype(float)
    weights  = 1.0 / counts[labels]
    sampler  = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"], sampler=sampler,
        num_workers=cfg["num_workers"], pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg["batch_size"] * 2, shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True,
    )
    return train_loader, val_loader


train_loader, val_loader = make_loaders(DATA_ROOT, CFG)


[train] real=4,816  AI=4,784  total=9,600
[val] real=1,184  AI=1,216  total=2,400
